[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/alg/u2_matrices.ipynb)

# Álg. U2 · Matrices y determinantes
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 2 del bloque de álgebra lineal. Tiene una sección por subtema, en el mismo orden que el libro:

1. Operaciones con matrices y transpuesta
2. Determinantes: Sarrus y cofactores
3. Propiedades y matriz inversa
4. Interpretación geométrica del determinante
5. Factorización LU

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una celda de teoría, una calculadora, sus casos de prueba y una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `ALG-U2` (bloque, unidad). Las celdas de código de cada sección vienen de `alg/u2_matrices/NN_*.py`; los fragmentos que el libro imprime, de `alg/u2_matrices/fragmentos/`.

**Convenciones.** Punto decimal. Una matriz se escribe **por renglones**: los renglones se separan con `;` o con un salto de línea, y las entradas de cada renglón con comas o espacios. Por ejemplo, `1, 2; 3, 4` es la matriz $\begin{bmatrix}1&2\\3&4\end{bmatrix}$ (también se acepta la forma de Python `[[1, 2], [3, 4]]`). Puedes escribir fracciones como `2/3`; los decimales se convierten a fracciones exactas (`0.1` se lee como $\tfrac{1}{10}$), así que las cuentas internas no acumulan error de redondeo. Un vector se escribe como lista, `1, 2, 3`, y se trata como columna. Los índices $i,j$ empiezan en 1, como en el libro. Los decimales que se muestran se *redondean* (no se truncan) a las cifras significativas que elijas.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
from decimal import Decimal, ROUND_HALF_UP
from fractions import Fraction
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, Math

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Utilidades compartidas por todo el notebook: leer matrices exactas, redondear y mostrar
_NUMERO = re.compile(r"[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?")
_AYUDA_NUMERO = "usa enteros, fracciones como 2/3 o decimales con punto"


_LIMITE = Fraction(10) ** 15                  # entradas entre 1e-15 y 1e15 en valor absoluto (o 0)


def _en_rango(q, texto):
    if q != 0 and not Fraction(1) / _LIMITE <= abs(q) <= _LIMITE:
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    return sp.Rational(q.numerator, q.denominator)


def numero(valor, nombre="el número"):
    """Un número como racional exacto de sympy: '0.1' -> 1/10, '2/3' -> 2/3, 5 -> 5, 1.2 -> 6/5."""
    if valor is None:
        raise ValueError(f"{nombre} está vacío: escribe un número.")
    if isinstance(valor, bool):
        raise ValueError(f"«{valor}» no es un número: {_AYUDA_NUMERO}.")
    if isinstance(valor, (int, np.integer)):
        return _en_rango(Fraction(int(valor)), valor)
    if isinstance(valor, (float, np.floating)) or (isinstance(valor, sp.Basic) and valor.is_Float):
        v = float(valor)
        if not math.isfinite(v):
            raise ValueError(f"{nombre} no es un número finito.")
        return _en_rango(Fraction(repr(v)), repr(v))      # repr da el decimal más corto: 0.1 -> '0.1' -> 1/10 exacto
    if isinstance(valor, sp.Basic):
        if valor.is_Rational:
            return valor                             # ya es exacto (por ejemplo, un resultado intermedio): no se acota
        raise ValueError(f"«{valor}» no es un número racional: {_AYUDA_NUMERO}.")
    texto = re.sub(r"\s+", "", str(valor)).replace("−", "-")      # acepta el signo menos tipográfico
    partes = texto.split("/")
    formas = [_NUMERO.fullmatch(p) for p in partes]
    if not texto or len(partes) > 2 or not all(formas):
        raise ValueError(f"«{str(valor).strip()}» no es un número: {_AYUDA_NUMERO}." if texto else
                         f"{nombre} está vacío: escribe un número.")
    if any(f.group(2) and abs(int(f.group(2)[1:])) > 30 for f in formas):   # 1e999999 no se calcula: se rechaza antes
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    num, *den = (Fraction(p) for p in partes)
    if den and den[0] == 0:
        raise ValueError(f"«{texto}» divide entre cero.")
    return _en_rango(num / den[0] if den else num, texto)


def _entradas(k):
    return f"{k} entrada" if k == 1 else f"{k} entradas"


def _desde_renglones(renglones, nombre):
    """Lista de renglones (cada uno, lista de entradas) -> Matrix de racionales, con mensajes claros."""
    if not renglones or not renglones[0]:
        raise ValueError(f"escribe {nombre} por renglones, por ejemplo: 1, 2; 3, 4")
    n = len(renglones[0])
    salida = []
    for i, renglon in enumerate(renglones, 1):
        if len(renglon) != n:
            raise ValueError(f"en {nombre}, el renglón {i} tiene {_entradas(len(renglon))} y el 1 tiene {n}: "
                             "todos los renglones deben tener el mismo número de entradas")
        fila = []
        for j, v in enumerate(renglon, 1):
            try:
                fila.append(numero(v))
            except ValueError as err:
                raise ValueError(f"en {nombre}, renglón {i}, columna {j}: {err}") from None
        salida.append(fila)
    return sp.Matrix(salida)


def matriz(texto, nombre="la matriz"):
    """Lee una matriz escrita por renglones ('1, 2; 3, 4' o un renglón por línea) como Matrix de racionales exactos."""
    if not isinstance(texto, str):
        return como_matriz(texto, nombre)
    t = texto.replace("−", "-")
    t = re.sub(r"[\]\)]\s*,?\s*[\[\(]", ";", t)            # [[1, 2], [3, 4]] o (1, 2), (3, 4): cada grupo es un renglón
    t = re.sub(r"[\[\]\(\)]", " ", t)
    renglones = [r.strip() for r in re.split(r"[;\n]", t)]
    renglones = [r for r in renglones if r]
    entradas = []
    for i, r in enumerate(renglones, 1):
        partes = re.split(r"\s*,\s*|\s+", r)
        if "" in partes:
            raise ValueError(f"en {nombre}, el renglón {i} tiene una entrada vacía: "
                             "revisa que no haya dos comas seguidas ni una coma al principio o al final")
        entradas.append(partes)
    return _desde_renglones(entradas, nombre)


def como_matriz(A, nombre="la matriz"):
    """Acepta texto, Matrix de sympy, lista de listas o arreglo de numpy y devuelve una Matrix de racionales exactos."""
    if isinstance(A, str):
        return matriz(A, nombre)
    if isinstance(A, sp.MatrixBase):
        renglones = A.tolist()
    elif isinstance(A, np.ndarray):
        if A.ndim > 2:
            raise ValueError(f"{nombre} debe ser una tabla de números (renglones y columnas)")
        renglones = np.atleast_2d(A).tolist()
    elif isinstance(A, (list, tuple)):
        anidada = [isinstance(f, (list, tuple, np.ndarray, sp.MatrixBase)) for f in A]
        if A and all(anidada):
            renglones = [list(f) for f in A]
        elif not any(anidada):
            renglones = [list(A)]                  # una lista sola es un renglón
        else:
            raise ValueError(f"en {nombre}, mezclaste números sueltos con renglones")
    else:
        renglones = [[A]]
    return _desde_renglones(renglones, nombre)


def vector(texto, nombre="el vector"):
    """Lee un vector ('1, 2, 3', '1; 2; 3', (1, 2, 3) o [1, 2, 3]) como columna de racionales exactos."""
    try:
        v = como_matriz(texto, nombre)
    except ValueError as err:
        if str(err).startswith("escribe "):          # vacío: el ejemplo debe ser de vector, no de matriz
            raise ValueError(f"escribe {nombre} como lista de números, por ejemplo: 1, 2, 3") from None
        raise
    if v.rows == 1 or v.cols == 1:
        return v.reshape(v.rows * v.cols, 1)
    raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 1, 2, 3 (escribiste una matriz {dims(v)})")


def dims(M):
    """Tamaño 'm×n' de una matriz."""
    return f"{M.rows}×{M.cols}"


def _cuadrada(A, mensaje="el determinante solo existe para matrices cuadradas"):
    if A.rows != A.cols:
        raise ValueError(f"{mensaje} (esta es {dims(A)})")


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    e = q.adjusted()
    if e > 15 or e < -4:                      # muy grandes o muy chicos: notación científica, 1.234×10^-7
        return f"{format(q.scaleb(-e), 'f')}×10^{e}"
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def _mensaje(fn):
    """Texto del ValueError que lanza fn() ('' si no lanza): para probar que el mensaje explica el error."""
    try:
        fn()
    except ValueError as err:
        return str(err)
    return ""


# --- formato de números y matrices ---------------------------------------------------------------
_SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")


def _sub(*indices):
    """Subíndices: _sub(1, 2) -> '₁₂'; con índices de dos cifras los separa con coma."""
    if all(k < 10 for k in indices):
        return "".join(str(k) for k in indices).translate(_SUB)
    return "₍" + ",".join(str(k) for k in indices).translate(_SUB) + "₎"


def _paren(v):
    """Un número listo para ir en un producto: negativos y fracciones entre paréntesis."""
    v = sp.sympify(v)
    return str(v) if (v >= 0 and v.is_integer) else f"({v})"


def _suma(valores):
    """'-16 - 5 + 0' en lugar de '-16 + -5 + 0'."""
    texto = str(valores[0])
    for v in valores[1:]:
        texto += f" - {-v}" if v < 0 else f" + {v}"
    return texto


def _en_linea(M):
    """Matriz en una línea con la misma notación de entrada: [2, 3; 5, -2]."""
    return "[" + "; ".join(", ".join(str(e) for e in M.row(i)) for i in range(M.rows)) + "]"


def _tupla(v, n=None):
    """Vector como (a, b, c); con n, además sus decimales redondeados a n cifras."""
    exacto = "(" + ", ".join(str(e) for e in v) + ")"
    if n is None or all(e.is_integer for e in v):
        return exacto
    return exacto + " ≈ (" + ", ".join(cifras(e, n) for e in v) + ")"


def _plano(M, fmt=str, corte=None):
    """Texto plano de una matriz con columnas alineadas; corte = columna tras la que se dibuja una barra."""
    celdas = [[fmt(M[i, j]) for j in range(M.cols)] for i in range(M.rows)]
    anchos = [max(len(celdas[i][j]) for i in range(M.rows)) for j in range(M.cols)]
    lineas = []
    for fila in celdas:
        partes = [c.rjust(a) for c, a in zip(fila, anchos)]
        if corte:
            partes = partes[:corte] + ["|"] + partes[corte:]
        lineas.append("[ " + "  ".join(partes) + " ]")
    return "\n".join(lineas)


_TEX = [("⁻¹", "^{-1}"), ("ᵀ", "^{T}"), ("·", r"\cdot "), ("×", r"\times "), ("≈", r"\approx "),
        ("−", "-"), ("det ", r"\det "), ("adj", r"\operatorname{adj}")]


def _a_tex(nombre):
    for a, b in _TEX:
        nombre = nombre.replace(a, b)
    return nombre


def _tex_decimales(M, n):
    filas = r" \\ ".join(" & ".join(cifras(M[i, j], n) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{matrix}" + filas + r"\end{matrix}\right]"


class _Formula(Math):
    """Math de IPython con una versión en texto plano (la que se ve donde no se dibuja LaTeX)."""
    def __init__(self, tex, texto):
        super().__init__(tex)
        self._texto = texto

    def __repr__(self):
        return self._texto


def muestra(nombre, M, n=None):
    """Muestra 'nombre = M' con LaTeX (con su tamaño si es matriz). Con n agrega los decimales redondeados a n cifras."""
    if isinstance(M, sp.MatrixBase):
        tex = f"{_a_tex(nombre)} = {sp.latex(M)}"
        texto = f"{nombre}  ({dims(M)}) =\n{_plano(M)}"
        if n is not None and not all(e.is_integer for e in M):
            tex += r" \approx " + _tex_decimales(M, n)
            texto += f"\n≈\n{_plano(M, lambda e: cifras(e, n))}"
        tex += rf"\qquad ({M.rows}\times {M.cols})"
    else:
        M = sp.sympify(M)
        tex, texto = f"{_a_tex(nombre)} = {sp.latex(M)}", f"{nombre} = {M}"
        if n is not None and not M.is_integer:
            tex += rf" \approx {cifras(M, n)}"
            texto += f" ≈ {cifras(M, n)}"
    display(_Formula(tex, texto))


def compara(mia, ref, pista=""):
    """'coinciden' o en qué entradas difiere tu matriz de la referencia, más una pista."""
    if mia.shape != ref.shape:
        return f"NO coinciden: tu matriz es {dims(mia)} y la de referencia es {dims(ref)}. {pista}".strip()
    malas = [f"({i + 1}, {j + 1})" for i in range(ref.rows) for j in range(ref.cols) if mia[i, j] != ref[i, j]]
    if not malas:
        return "coinciden"
    cuales = "la entrada" if len(malas) == 1 else "las entradas"
    return f"NO coinciden en {cuales} {', '.join(malas)} (renglón, columna). {pista}".strip()

## 1. Operaciones con matrices y transpuesta

Una matriz $A$ de tamaño $m\times n$ tiene $m$ renglones y $n$ columnas; su entrada $a_{ij}$ está en el renglón $i$ y la columna $j$.

**Suma y resta** (solo entre matrices del mismo tamaño): $(A\pm B)_{ij}=a_{ij}\pm b_{ij}$. **Múltiplo escalar:** $(kA)_{ij}=k\,a_{ij}$.

**Producto.** Si $A$ es $m\times n$ y $B$ es $n\times p$, la entrada $c_{ij}$ de $C=AB$ combina el renglón $i$ de $A$ con la columna $j$ de $B$:

$$c_{ij}=\sum_{k=1}^{n}a_{ik}\,b_{kj}=a_{i1}b_{1j}+a_{i2}b_{2j}+\cdots+a_{in}b_{nj},\qquad (m\times n)(n\times p)=m\times p.$$

Las columnas de $A$ deben ser tantas como los renglones de $B$ (los dos $n$ de en medio); los números de los extremos dan el tamaño del resultado.

- **Identidad** $I_n$: unos en la diagonal y ceros fuera de ella. Si $A$ es $m\times n$, $I_mA=AI_n=A$, como el 1 entre los números.
- **Transpuesta:** $(A^T)_{ij}=a_{ji}$; los renglones pasan a ser columnas. Cumple $(A^T)^T=A$, $(A+B)^T=A^T+B^T$ y $(AB)^T=B^TA^T$: al transponer un producto, **el orden se invierte**.
- En general $AB\neq BA$. A veces los dos productos existen y dan matrices distintas; a veces uno de ellos ni siquiera está definido.

**Ejemplo: materiales y costo.** Una planta fabrica tres productos con tres materiales. La entrada $m_{ij}$ de $M$ es la cantidad del material $i$ que lleva una unidad del producto $j$, y la entrada $p_{jk}$ de $P$ es el número de unidades del producto $j$ que se fabrican en la semana $k$. Entonces $MP$ ($3\times 3$ por $3\times 2$, resultado $3\times 2$) da el material $i$ que se consume en la semana $k$, y con el vector $\mathbf c$ de costos por unidad de material, el renglón $\mathbf c^T(MP)$ da el costo de material de cada semana:

$$M=\begin{bmatrix}0.8&2.5&4\\12&20&32\\0.2&0.6&0.9\end{bmatrix},\quad P=\begin{bmatrix}40&25\\10&15\\6&8\end{bmatrix},\quad MP=\begin{bmatrix}81&89.5\\872&856\\19.4&21.2\end{bmatrix},\quad \mathbf c=\begin{bmatrix}180\\0.5\\120\end{bmatrix},\quad \mathbf c^T(MP)=\begin{bmatrix}17\,344&19\,082\end{bmatrix}.$$

Supuesto del ejemplo: cada cantidad de $M$ va en la unidad propia de su material y $\mathbf c$ está en pesos por esa unidad, así que $\mathbf c^T(MP)$ queda en pesos. Para reproducirlo en la calculadora escribe A = `0.8, 2.5, 4; 12, 20, 32; 0.2, 0.6, 0.9`, B = `40, 25; 10, 15; 6, 8` y elige la operación AB.

In [ ]:
def suma(A, B, nombres=("A", "B")):
    """A + B exacta; ValueError si las dos matrices no tienen el mismo tamaño."""
    a, b = nombres
    A, B = como_matriz(A, a), como_matriz(B, b)
    if A.shape != B.shape:
        raise ValueError(f"{a} es {dims(A)} y {b} es {dims(B)}: para sumar o restar, las dos deben tener el mismo tamaño")
    return A + B


def producto(A, B, nombres=("A", "B")):
    """AB exacto; ValueError que explica el tamaño si las columnas de A no son tantas como los renglones de B."""
    a, b = nombres
    A, B = como_matriz(A, a), como_matriz(B, b)
    if A.cols != B.rows:
        raise ValueError(f"{a} es {dims(A)} y {b} es {dims(B)}: para {a}{b}, las columnas de {a} ({A.cols}) "
                         f"deben ser tantas como los renglones de {b} ({B.rows})")
    return A * B


def transpuesta(A):
    """Aᵀ: el renglón i de A pasa a ser la columna i."""
    return como_matriz(A).T


def entrada_del_producto(A, B, i=1, j=1, nombres=("A", "B")):
    """Texto con la cuenta de la entrada (i, j) de AB: renglón i de A por columna j de B."""
    C = producto(A, B, nombres)
    A, B = como_matriz(A), como_matriz(B)
    terminos = " + ".join(f"{_paren(A[i - 1, k])}·{_paren(B[k, j - 1])}" for k in range(A.cols))
    return f"c{_sub(i, j)} = (renglón {i} de {nombres[0]})·(columna {j} de {nombres[1]}) = {terminos} = {C[i - 1, j - 1]}"


def ab_y_ba(A, B):
    """(AB o None, BA o None, veredicto en palabras)."""
    A, B = como_matriz(A, "A"), como_matriz(B, "B")
    try:
        AB = producto(A, B)
    except ValueError as err:
        AB, por_que_ab = None, str(err)
    try:
        BA = producto(B, A, ("B", "A"))
    except ValueError as err:
        BA, por_que_ba = None, str(err)
    if AB is None and BA is None:
        return AB, BA, f"ni AB ni BA están definidos: A es {dims(A)} y B es {dims(B)}"
    if BA is None:
        return AB, BA, f"AB sí está definido ({dims(AB)}), pero BA no: {por_que_ba}"
    if AB is None:
        return AB, BA, f"BA sí está definido ({dims(BA)}), pero AB no: {por_que_ab}"
    if AB.shape != BA.shape:
        return AB, BA, f"AB es {dims(AB)} y BA es {dims(BA)}: ni siquiera tienen el mismo tamaño, así que AB ≠ BA"
    if AB == BA:
        return AB, BA, "AB = BA: estas dos matrices conmutan (es la excepción, no la regla)"
    return AB, BA, "AB ≠ BA: el orden de los factores sí importa"


def _muestra_producto(A, B, nombres, n):
    C = producto(A, B, nombres)
    muestra("".join(nombres), C, n)
    print("Por ejemplo,", entrada_del_producto(A, B, 1, 1, nombres))


_OPERACIONES = ["A + B", "A − B", "AB", "BA", "AB y BA", "Aᵀ", "(AB)ᵀ y BᵀAᵀ"]


def calculadora_operaciones(A_txt, B_txt, operacion, n_cifras):
    try:
        A = matriz(A_txt, "A")
        B = None if operacion == "Aᵀ" else matriz(B_txt, "B")      # Aᵀ no necesita a B
        muestra("A", A, n_cifras)
        if B is not None:
            muestra("B", B, n_cifras)
        if operacion == "A + B":
            muestra("A + B", suma(A, B), n_cifras)
        elif operacion == "A − B":
            muestra("A − B", suma(A, -B), n_cifras)
        elif operacion == "AB":
            _muestra_producto(A, B, ("A", "B"), n_cifras)
        elif operacion == "BA":
            _muestra_producto(B, A, ("B", "A"), n_cifras)
        elif operacion == "AB y BA":
            AB, BA, veredicto = ab_y_ba(A, B)
            if AB is not None:
                muestra("AB", AB, n_cifras)
            if BA is not None:
                muestra("BA", BA, n_cifras)
            print(veredicto)
        elif operacion == "Aᵀ":
            muestra("Aᵀ", transpuesta(A), n_cifras)
            print(f"A es {dims(A)} y Aᵀ es {dims(transpuesta(A))}: el renglón 1 de A es la columna 1 de Aᵀ.")
        else:
            izquierda = transpuesta(producto(A, B))
            derecha = producto(transpuesta(B), transpuesta(A), ("Bᵀ", "Aᵀ"))
            muestra("(AB)ᵀ", izquierda, n_cifras)
            muestra("BᵀAᵀ", derecha, n_cifras)
            print("(AB)ᵀ = BᵀAᵀ: coinciden, como dice la regla." if izquierda == derecha else "NO coinciden: revisa la entrada.")
            try:
                AtBt = producto(transpuesta(A), transpuesta(B), ("Aᵀ", "Bᵀ"))
                if AtBt.shape != izquierda.shape:
                    print(f"Ojo: AᵀBᵀ (sin invertir el orden) es {dims(AtBt)}, ni siquiera del tamaño de (AB)ᵀ.")
                elif AtBt != izquierda:
                    print("Ojo: AᵀBᵀ (sin invertir el orden) da otra matriz:")
                    muestra("AᵀBᵀ", AtBt, n_cifras)
                else:
                    print("Aquí AᵀBᵀ también coincide, pero es casualidad: en general hay que invertir el orden.")
            except ValueError as err:
                print(f"Ojo: AᵀBᵀ (sin invertir el orden) ni siquiera está definido: {err}")
    except ValueError as err:
        print("Revisa la entrada:", err)


widgets.interact(calculadora_operaciones,
    A_txt=widgets.Text(value="1, 2; 3, 4", description="A =", continuous_update=False),
    B_txt=widgets.Text(value="0, 1; 1, 0", description="B =", continuous_update=False),
    operacion=widgets.Dropdown(options=_OPERACIONES, value="AB y BA", description="operación"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
_A1, _B1 = matriz("1, 2; 3, 4"), matriz("0, 1; 1, 0")
_A23, _B32 = matriz("2, 0, 1; 1, -1, 3"), matriz("1, 2; 0, 1; 4, -2")
_M, _P = matriz("0.8, 2.5, 4; 12, 20, 32; 0.2, 0.6, 0.9"), matriz("40, 25; 10, 15; 6, 8")
PRUEBAS_1 = [
    ("[[1,2],[3,4]]·[[0,1],[1,0]] = [[2,1],[4,3]] y la cuenta de c₁₂ es 1·1 + 2·0 = 1",
     lambda: producto(_A1, _B1) == sp.Matrix([[2, 1], [4, 3]])
             and entrada_del_producto(_A1, _B1, 1, 2).endswith("= 1·1 + 2·0 = 1")),
    ("BA = [[3,4],[1,2]] y BA ≠ AB", lambda: producto(_B1, _A1) == sp.Matrix([[3, 4], [1, 2]])
                                            and ab_y_ba(_A1, _B1)[2].startswith("AB ≠ BA")),
    ("(2×3)(3×2) da una 2×2", lambda: dims(producto(_A23, _B32)) == "2×2"),
    ("(2×3)(2×3) se rechaza y el mensaje explica el tamaño",
     lambda: _mensaje(lambda: producto(_A23, _A23)) == "A es 2×3 y B es 2×3: para AB, las columnas de A (3) "
                                                     "deben ser tantas como los renglones de B (2)"),
    ("(AB)ᵀ = BᵀAᵀ", lambda: transpuesta(producto(_A23, _B32)) == producto(transpuesta(_B32), transpuesta(_A23))),
    ("materiales: MP = [[81, 89.5], [872, 856], [19.4, 21.2]]",
     lambda: producto(_M, _P) == matriz("81, 89.5; 872, 856; 19.4, 21.2")),
    ("costo por semana cᵀ(MP) = (17344, 19082)",
     lambda: producto(transpuesta(vector("180, 0.5, 120")), producto(_M, _P)) == sp.Matrix([[17344, 19082]])),
    ("A + B = [[1,3],[4,4]] y A + B de tamaños distintos se rechaza",
     lambda: suma(_A1, _B1) == sp.Matrix([[1, 3], [4, 4]]) and _rechaza(lambda: suma(_A1, _A23))),
    ("lectura exacta: 0.1 -> 1/10 y 2/3 queda como 2/3",
     lambda: matriz("0.1, 2/3")[0, 0] == sp.Rational(1, 10) and matriz("0.1 2/3")[0, 1] == sp.Rational(2, 3)),
    ("renglones disparejos, texto vacío y letras se rechazan",
     lambda: "el renglón 2 tiene 1 entrada y el 1 tiene 2" in _mensaje(lambda: matriz("1, 2; 3"))
             and _rechaza(lambda: matriz("  ")) and _rechaza(lambda: matriz("1, x; 3, 4"))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano $\begin{bmatrix}1&-1\\2&0\end{bmatrix}\begin{bmatrix}3&1\\1&2\end{bmatrix}$ y escribe el resultado como texto, renglón por renglón.

In [ ]:
mi_AB = None     # escribe tu resultado como texto, por ejemplo: "1, 0; 0, 1"

if mi_AB is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mia = matriz(str(mi_AB), "tu resultado")
    except ValueError as err:
        print("Revisa tu matriz:", err)
    else:
        ref = producto(matriz("1, -1; 2, 0"), matriz("3, 1; 1, 2"))
        print("La calculadora da:")
        muestra("AB", ref)
        print(compara(mia, ref, "Cada c_ij es el renglón i de la primera matriz por la columna j de la segunda; "
                                "por ejemplo, c₁₂ = a₁₁b₁₂ + a₁₂b₂₂."))

## 2. Determinantes: Sarrus y cofactores

El determinante es un número asociado a una matriz **cuadrada**. Para una $2\times 2$:

$$\det\begin{bmatrix}a&b\\c&d\end{bmatrix}=ad-bc.$$

**Menores y cofactores.** El menor $M_{ij}$ es el determinante de la submatriz que queda al borrar el renglón $i$ y la columna $j$. El cofactor le agrega un signo que alterna como un tablero de ajedrez:

$$C_{ij}=(-1)^{i+j}M_{ij},\qquad \text{signos: }\begin{bmatrix}+&-&+\\-&+&-\\+&-&+\end{bmatrix}.$$

**Desarrollo por cofactores.** Se hace por cualquier renglón $i$ o por cualquier columna $j$:

$$\det A=\sum_{j=1}^{n}a_{ij}C_{ij}\quad(\text{renglón } i)\qquad\text{o}\qquad \det A=\sum_{i=1}^{n}a_{ij}C_{ij}\quad(\text{columna } j).$$

Todas las líneas dan el mismo número, así que conviene la que tiene más ceros: cada $a_{ij}=0$ elimina un término completo, con todo y su cofactor.

**Regla de Sarrus (solo $3\times 3$).** Copia las dos primeras columnas a la derecha de la matriz; suma los productos de las tres diagonales que bajan y resta los de las tres que suben:

$$\det A=a_{11}a_{22}a_{33}+a_{12}a_{23}a_{31}+a_{13}a_{21}a_{32}-a_{13}a_{22}a_{31}-a_{11}a_{23}a_{32}-a_{12}a_{21}a_{33}.$$

No se generaliza: un determinante $4\times 4$ tiene $4!=24$ términos y las "diagonales" de Sarrus solo darían 8.

**Costo.** Sin ceros, el desarrollo por cofactores de una $n\times n$ hace $\sum_{k=1}^{n-1} n!/k!\approx 1.7\,n!$ multiplicaciones: 9 para $n=3$, 40 para $n=4$, 205 para $n=5$ y unos 6.2 millones para $n=10$. Por eso el software no calcula determinantes así, sino por eliminación (sección 5), que hace del orden de $n^3/3$.

La calculadora muestra qué renglón o columna usa en cada nivel, cada término $a_{ij}C_{ij}$, cuántas multiplicaciones hizo (un término con $a_{ij}=0$ no cuenta) y lo compara con el determinante exacto de `sympy`.

In [ ]:
def _indice(k, n, letra):
    if isinstance(k, bool) or not isinstance(k, (int, np.integer, sp.Integer)) or not 1 <= int(k) <= n:
        raise ValueError(f"{letra} = {k} no sirve: i y j van de 1 a {n} (la matriz es {n}×{n})")
    return int(k)


def submatriz(A, i, j):
    """A sin el renglón i ni la columna j (i, j desde 1)."""
    A = como_matriz(A, "A")
    _cuadrada(A, "los menores solo existen para matrices cuadradas")
    if A.rows < 2:
        raise ValueError("una matriz 1×1 no tiene menores: su determinante es su única entrada")
    i, j = _indice(i, A.rows, "i"), _indice(j, A.rows, "j")
    return A.minor_submatrix(i - 1, j - 1)


def menor(A, i, j):
    """M_ij: determinante de la submatriz sin el renglón i ni la columna j."""
    return submatriz(A, i, j).det()


def cofactor(A, i, j):
    """C_ij = (-1)^(i+j) M_ij."""
    return (-1) ** (int(i) + int(j)) * menor(A, i, j)


_BAJAN = [((1, 1), (2, 2), (3, 3)), ((1, 2), (2, 3), (3, 1)), ((1, 3), (2, 1), (3, 2))]
_SUBEN = [((1, 3), (2, 2), (3, 1)), ((1, 1), (2, 3), (3, 2)), ((1, 2), (2, 1), (3, 3))]


def det_sarrus(A):
    """(det A, texto con los pasos) por la regla de Sarrus; solo para 3×3."""
    A = como_matriz(A, "A")
    if A.shape != (3, 3):
        raise ValueError(f"la regla de Sarrus solo vale para matrices 3×3 (esta es {dims(A)}); usa cofactores")
    a = lambda p: A[p[0] - 1, p[1] - 1]
    cuenta = lambda d: "·".join(_paren(a(p)) for p in d)
    bajan = [a(d[0]) * a(d[1]) * a(d[2]) for d in _BAJAN]
    suben = [a(d[0]) * a(d[1]) * a(d[2]) for d in _SUBEN]
    det = sum(bajan) - sum(suben)
    pasos = "\n".join([
        "Copia las dos primeras columnas a la derecha de A:",
        _plano(A.row_join(A[:, :2]), corte=3),
        f"Diagonales que bajan (+): {' + '.join(cuenta(d) for d in _BAJAN)} = {_suma(bajan)} = {sum(bajan)}",
        f"Diagonales que suben (-): {' + '.join(cuenta(d) for d in _SUBEN)} = {_suma(suben)} = {sum(suben)}",
        f"det A = {sum(bajan)} - {_paren(sum(suben))} = {det}",
    ])
    return det, pasos


def _mejor_linea(A):
    """('renglón' o 'columna', índice desde 1, ceros) de la línea con más ceros; en empate gana la primera."""
    mejor = ("renglón", 1, -1)
    for tipo, lineas in (("renglón", [A.row(i) for i in range(A.rows)]), ("columna", [A.col(j) for j in range(A.cols)])):
        for k, linea in enumerate(lineas, 1):
            ceros = sum(1 for e in linea if e == 0)
            if ceros > mejor[2]:
                mejor = (tipo, k, ceros)
    return mejor


def _expande(A, nivel, pasos, nombre):
    """(det, multiplicaciones) por cofactores; si pasos es una lista, le agrega el texto de cada paso."""
    n, s = A.rows, "    " * nivel
    if n == 1:
        if pasos is not None and nivel == 0:
            pasos.append(f"det {nombre} = {A[0, 0]} (una matriz 1×1: su determinante es su única entrada)")
        return A[0, 0], 0
    if n == 2:
        a, b, c, d = A
        det = a * d - b * c
        if pasos is not None:
            pasos.append(f"{s}det {nombre} = {_paren(a)}·{_paren(d)} - {_paren(b)}·{_paren(c)} = {det}")
        return det, int(a != 0 and d != 0) + int(b != 0 and c != 0)
    tipo, k, ceros = _mejor_linea(A)
    if pasos is not None:
        nota = f" (tiene {ceros} cero{'s' if ceros != 1 else ''})" if ceros else ""
        indices = " (índices de la submatriz)" if nivel > 0 else ""      # a_ij y C_ij de abajo cuentan dentro del menor
        pasos.append(f"{s}det {nombre} ({dims(A)}): desarrollo por el {tipo} {k}{nota}{indices}")
    total, mult, terminos = 0, 0, []
    for t in range(1, n + 1):
        i, j = (k, t) if tipo == "renglón" else (t, k)
        a = A[i - 1, j - 1]
        if a == 0:
            if pasos is not None:
                pasos.append(f"{s}  a{_sub(i, j)} = 0: ese término vale 0 y su cofactor no se calcula")
            continue
        signo = "+1" if (i + j) % 2 == 0 else "-1"
        sub = A.minor_submatrix(i - 1, j - 1)
        if pasos is not None:
            pasos.append(f"{s}  a{_sub(i, j)}·C{_sub(i, j)} = {_paren(a)}·({signo})·M{_sub(i, j)},  M{_sub(i, j)} = det {_en_linea(sub)}")
        M, m = _expande(sub, nivel + 1, pasos, _en_linea(sub))
        termino = a * (-1) ** (i + j) * M
        mult += m + 1                                  # las del menor y la de a_ij·C_ij
        total += termino
        terminos.append(termino)
        if pasos is not None:
            pasos.append(f"{s}  a{_sub(i, j)}·C{_sub(i, j)} = {_paren(a)}·({signo})·{_paren(M)} = {termino}")
    if pasos is not None:
        if len(terminos) > 1:
            pasos.append(f"{s}det {nombre} = {_suma(terminos)} = {total}")
        elif terminos:
            pasos.append(f"{s}det {nombre} = {total} (solo un término no es cero)")
        else:
            pasos.append(f"{s}det {nombre} = 0: todas las entradas de esa línea son cero")
    return total, mult


def det_cofactores(A, pasos=False):
    """(det A, lista de pasos, multiplicaciones) por desarrollo de cofactores, siempre por la línea con más ceros."""
    A = como_matriz(A, "A")
    _cuadrada(A)
    lista = [] if pasos else None
    det, mult = _expande(A, 0, lista, "A")
    return det, (lista or []), mult


def multiplicaciones_sin_ceros(n):
    """Multiplicaciones del desarrollo por cofactores de una n×n sin ceros: f(n) = n(1 + f(n-1)), f(2) = 2."""
    f = 0
    for k in range(2, n + 1):
        f = k * (1 + f) if k > 2 else 2
    return f


_MAX_COFACTORES = 7      # 8! = 40320 términos: más que eso ya tarda demasiado
_MAX_PASOS = 80          # renglones de pasos que se imprimen completos


def calculadora_determinante(A_txt, metodo, n_cifras):
    try:
        A = matriz(A_txt, "A")
        _cuadrada(A)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    muestra("A", A, n_cifras)
    exacto = A.det()
    calculados = []
    if metodo in ("Sarrus", "los dos"):
        try:
            d, texto = det_sarrus(A)
        except ValueError as err:
            if metodo == "Sarrus":
                print("Revisa la entrada:", err); return
            print("Sarrus no aplica:", err)
        else:
            print("— Regla de Sarrus —")
            print(texto)
            print("Multiplicaciones: 12 (6 productos de 3 factores, aunque haya ceros)")
            calculados.append(("Sarrus", d))
    if metodo in ("cofactores", "los dos"):
        if A.rows > _MAX_COFACTORES:
            print(f"Cofactores: con una {dims(A)} serían hasta {multiplicaciones_sin_ceros(A.rows)} multiplicaciones; "
                  f"esta calculadora los desarrolla hasta {_MAX_COFACTORES}×{_MAX_COFACTORES}. Usa la factorización LU (sección 5).")
            print(f"det A (sympy) = {exacto}" + ("" if exacto.is_integer else f" ≈ {cifras(exacto, n_cifras)}"))
        else:
            d, pasos, mult = det_cofactores(A, pasos=True)
            print("— Desarrollo por cofactores —")
            if len(pasos) <= _MAX_PASOS:
                print("\n".join(pasos))
            else:
                print("\n".join(p for p in pasos if not p.startswith("    ")))
                print(f"(se omiten {sum(1 for p in pasos if p.startswith('    '))} renglones de los menores internos)")
            print(f"Multiplicaciones: {mult} (una {dims(A)} sin ceros pediría {multiplicaciones_sin_ceros(A.rows)})")
            calculados.append(("cofactores", d))
    for metodo_usado, d in calculados:
        print(f"{metodo_usado}: det A = {d}" + ("" if d.is_integer else f" ≈ {cifras(d, n_cifras)}") +
              (" | coincide con sympy" if d == exacto else f" | NO coincide con sympy ({exacto})"))


widgets.interact(calculadora_determinante,
    A_txt=widgets.Textarea(value="2, -1, 3\n0, 4, 1\n5, 2, -2", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="320px", height="110px")),
    metodo=widgets.Dropdown(options=["cofactores", "Sarrus", "los dos"], value="los dos", description="método"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
_D3 = matriz("2, -1, 3; 0, 4, 1; 5, 2, -2")
_D4 = matriz("1, 0, 2, 0; 0, 3, 0, 1; 2, 0, 1, 0; 0, 1, 0, 2")
PRUEBAS_2 = [
    ("det [[3,1],[4,2]] = 2", lambda: det_cofactores(matriz("3, 1; 4, 2"))[0] == 2),
    ("det [[2,-1,3],[0,4,1],[5,2,-2]] = -85 por Sarrus y por cofactores",
     lambda: det_sarrus(_D3)[0] == -85 and det_cofactores(_D3)[0] == -85),
    ("en esa matriz, M₁₂ = -5 y C₁₂ = 5", lambda: menor(_D3, 1, 2) == -5 and cofactor(_D3, 1, 2) == 5),
    ("4×4 por cofactores = det de sympy (-15)", lambda: det_cofactores(_D4)[0] == _D4.det() == -15),
    ("3×3 sin ceros: 9 multiplicaciones; con un cero, menos",
     lambda: det_cofactores(matriz("1, 2, 3; 4, 5, 6; 7, 8, 10"))[2] == 9 == multiplicaciones_sin_ceros(3)
             and det_cofactores(_D3)[2] < 9),
    ("Sarrus en una 4×4 se rechaza", lambda: "solo vale para matrices 3×3" in _mensaje(lambda: det_sarrus(_D4))),
    ("una matriz no cuadrada se rechaza", lambda: _rechaza(lambda: det_cofactores(matriz("1, 2, 3; 4, 5, 6")))),
    ("i o j fuera de rango se rechazan", lambda: _rechaza(lambda: menor(_D3, 4, 1)) and _rechaza(lambda: cofactor(_D3, 0, 2))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano $\det\begin{bmatrix}1&2&0\\3&-1&2\\0&1&1\end{bmatrix}$ por Sarrus y por cofactores; los dos deben darte el mismo número. Escríbelo.

In [ ]:
mi_det = None     # escribe un número (también sirve como texto, por ejemplo "-7")

if mi_det is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = numero(mi_det, "tu determinante")
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = det_cofactores(matriz("1, 2, 0; 3, -1, 2; 0, 1, 1"))[0]
        print("La calculadora da: det =", ref)
        print("coinciden" if mio == ref else
              "NO coinciden: desarrolla por el renglón 1 o la columna 1 (tienen un cero) y cuida el signo (-1)^(i+j) de cada cofactor")

## 3. Propiedades y matriz inversa

**Propiedades del determinante** ($A$ y $B$ de $n\times n$, $k$ un número):

- $\det(AB)=\det A\cdot\det B$ y $\det(A^T)=\det A$.
- $\det(kA)=k^n\det A$: el factor $k$ sale una vez **por renglón**, no una sola vez.
- Operaciones de renglón: intercambiar dos renglones cambia el signo; multiplicar un renglón por $k$ multiplica el determinante por $k$; sumar a un renglón un múltiplo de otro no lo cambia.

**Inversa.** $A^{-1}$ es la matriz que cumple $AA^{-1}=A^{-1}A=I$. Existe **si y solo si** $\det A\neq 0$. Para una $2\times 2$:

$$\begin{bmatrix}a&b\\c&d\end{bmatrix}^{-1}=\frac{1}{ad-bc}\begin{bmatrix}d&-b\\-c&a\end{bmatrix}.$$

En general, con la matriz de cofactores $C=[C_{ij}]$ y la **adjunta** $\operatorname{adj}(A)=C^T$ (la transpuesta de la matriz de cofactores):

$$A^{-1}=\frac{\operatorname{adj}(A)}{\det A},\qquad (AB)^{-1}=B^{-1}A^{-1}.$$

Igual que con la transpuesta, la inversa de un producto **invierte el orden**. Si $y=Ax$, entonces $x=A^{-1}y$. Después de calcular una inversa a mano, comprueba siempre que $AA^{-1}=I$: es la forma más rápida de atrapar un signo o una transposición olvidada.

In [ ]:
def matriz_cofactores(A):
    """C = [C_ij], con C_ij = (-1)^(i+j) M_ij."""
    A = como_matriz(A, "A")
    _cuadrada(A, "la matriz de cofactores solo existe para matrices cuadradas")
    n = A.rows
    if n == 1:
        return sp.Matrix([[1]])                     # convención: adj de una 1×1 es [1]
    return sp.Matrix(n, n, lambda i, j: cofactor(A, int(i) + 1, int(j) + 1))


def adjunta(A):
    """adj(A) = Cᵀ, la transpuesta de la matriz de cofactores."""
    return matriz_cofactores(A).T


def inversa(A):
    """(A⁻¹, det A) exactas por A⁻¹ = adj(A)/det A (hasta 7×7; más grande, con A.inv() de sympy); ValueError si det A = 0."""
    A = como_matriz(A, "A")
    _cuadrada(A, "la inversa solo existe para matrices cuadradas")
    d = A.det()
    if d == 0:
        raise ValueError("det A = 0: la matriz no tiene inversa")
    if A.rows > _MAX_COFACTORES:                      # n² cofactores de (n-1)×(n-1) ya no valen la pena
        return A.inv(), d
    return adjunta(A) / d, d


def es_identidad(M):
    """True si M es la matriz identidad."""
    M = como_matriz(M)
    return M.rows == M.cols and M == sp.eye(M.rows)


def vector_nulo(A):
    """Un x ≠ 0 con A x = 0 (existe cuando det A = 0), con entradas enteras."""
    A = como_matriz(A, "A")
    base = A.nullspace()
    if not base:
        raise ValueError("A x = 0 solo tiene la solución x = 0 (det A ≠ 0)")
    v = base[0]
    v = v * math.lcm(*[int(e.q) for e in v])         # sin fracciones
    v = v / math.gcd(*[int(e.p) for e in v])         # enteros lo más chicos posible
    primero = next(e for e in v if e != 0)
    return v if primero > 0 else -v                  # la primera entrada distinta de cero, positiva


def despeja(A, y):
    """x = A⁻¹ y, la solución exacta de y = A x."""
    A, y = como_matriz(A, "A"), vector(y, "y")
    _cuadrada(A, "para despejar x con A⁻¹, A debe ser cuadrada")
    if y.rows != A.rows:
        raise ValueError(f"y tiene {_entradas(y.rows)} y A tiene {A.rows} renglones: deben coincidir")
    try:
        Ainv, _ = inversa(A)
    except ValueError as err:
        raise ValueError(f"{err}, así que y = A x no tiene solución única "
                         "(puede no tener ninguna o tener infinitas; lo verás en la unidad de sistemas)") from None
    return Ainv * y


def calculadora_inversa(A_txt, n_cifras):
    try:
        A = matriz(A_txt, "A")
        _cuadrada(A, "la inversa solo existe para matrices cuadradas")
    except ValueError as err:
        print("Revisa la entrada:", err); return
    muestra("A", A, n_cifras)
    d = A.det()
    muestra("det A", d, n_cifras)
    if d == 0:
        print("det A = 0: A no tiene inversa. Hay un vector x ≠ 0 que A manda al vector cero, igual que manda 0 a 0;")
        print("como dos entradas distintas dan la misma salida, no hay forma de deshacer A. Por ejemplo:")
        x = vector_nulo(A)
        muestra("x", x)
        muestra("A x", A * x)
        return
    if A.rows > _MAX_COFACTORES:
        print(f"Con una {dims(A)}, la matriz de cofactores pide {A.rows ** 2} determinantes de "
              f"{A.rows - 1}×{A.rows - 1}; aquí se muestran hasta {_MAX_COFACTORES}×{_MAX_COFACTORES}.")
        print("La inversa se calcula con A.inv() de sympy (eliminación, también exacta).")
    else:
        print("Matriz de cofactores, C_ij = (-1)^(i+j)·M_ij:")
        muestra("C", matriz_cofactores(A))
        print("Adjunta: la transpuesta de C.")
        muestra("adj(A)", adjunta(A))
    Ainv, _ = inversa(A)
    if A.rows <= _MAX_COFACTORES:
        print("Inversa: la adjunta dividida entre det A.")
    muestra("A⁻¹", Ainv, n_cifras)
    P = A * Ainv
    muestra("A·A⁻¹", P)
    print("Verificación: A·A⁻¹ = I" if es_identidad(P) else "Verificación FALLÓ: A·A⁻¹ ≠ I")


widgets.interact(calculadora_inversa,
    A_txt=widgets.Textarea(value="2, 1, 0\n0, 2, 1\n1, 0, 2", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="320px", height="110px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

**Despeja $x$ de $y=Ax$.** Si conoces la salida $y$ de un sistema lineal y la matriz $A$ que lo describe, la entrada que la produjo es $x=A^{-1}y$. La calculadora la obtiene exacta y comprueba que $Ax=y$.

In [ ]:
def calculadora_despeje(A_txt, y_txt, n_cifras):
    try:
        A, y = matriz(A_txt, "A"), vector(y_txt, "y")
        x = despeja(A, y)
    except ValueError as err:     # det A = 0 no es un error de captura: el sistema no tiene solución única
        print("No hay solución única:" if str(err).startswith("det A = 0") else "Revisa la entrada:", err); return
    muestra("A", A, n_cifras)
    muestra("y", y, n_cifras)
    muestra("x = A⁻¹y", x, n_cifras)
    muestra("A x", A * x, n_cifras)
    print("Verificación: A x = y" if A * x == y else "Verificación FALLÓ: A x ≠ y")


widgets.interact(calculadora_despeje,
    A_txt=widgets.Textarea(value="2, 1, 0\n0, 2, 1\n1, 0, 2", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="320px", height="110px")),
    y_txt=widgets.Text(value="11, 12, 7", description="y =", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
_S = matriz("2, 1, 0; 0, 2, 1; 1, 0, 2")
_A3, _B3 = matriz("1, 2; 3, 5"), matriz("2, 1; 1, 1")
PRUEBAS_3 = [
    ("inversa de [[2,1],[5,3]] = [[3,-1],[-5,2]]", lambda: inversa(matriz("2, 1; 5, 3"))[0] == sp.Matrix([[3, -1], [-5, 2]])),
    ("S = [[2,1,0],[0,2,1],[1,0,2]]: det 9 y adj S = [[4,-2,1],[1,4,-2],[-2,1,4]]",
     lambda: inversa(_S)[1] == 9 and adjunta(_S) == sp.Matrix([[4, -2, 1], [1, 4, -2], [-2, 1, 4]])),
    ("S⁻¹·(11, 12, 7) = (3, 5, 2), S·S⁻¹ = I y S no es la identidad",
     lambda: despeja(_S, "11, 12, 7") == vector("3, 5, 2") and es_identidad(_S * inversa(_S)[0]) and not es_identidad(_S)),
    ("[[1,2],[2,4]] (det 0) se rechaza", lambda: _mensaje(lambda: inversa(matriz("1, 2; 2, 4"))) ==
                                                "det A = 0: la matriz no tiene inversa"),
    ("con det 0, A x = 0 tiene un x ≠ 0, proporcional a (2, -1)",
     lambda: (lambda v: v[0] != 0 and 2 * v == v[0] * vector("2, -1") and matriz("1, 2; 2, 4") * v == sp.zeros(2, 1))(
             vector_nulo(matriz("1, 2; 2, 4")))),
    ("(AB)⁻¹ = B⁻¹A⁻¹ (y ≠ A⁻¹B⁻¹)",
     lambda: inversa(producto(_A3, _B3))[0] == producto(inversa(_B3)[0], inversa(_A3)[0])
             != producto(inversa(_A3)[0], inversa(_B3)[0])),
    ("det(2S) = 2³·det S = 72", lambda: det_cofactores(2 * _S)[0] == 8 * det_cofactores(_S)[0] == 72),
    ("det(AB) = det A·det B y det(Aᵀ) = det A",
     lambda: det_cofactores(producto(_A3, _B3))[0] == det_cofactores(_A3)[0] * det_cofactores(_B3)[0]
             and det_cofactores(transpuesta(_S))[0] == det_cofactores(_S)[0]),
    ("1×1: [5]⁻¹ = [1/5]; una no cuadrada se rechaza",
     lambda: inversa(matriz("5"))[0] == sp.Matrix([[sp.Rational(1, 5)]]) and _rechaza(lambda: inversa(matriz("1, 2, 3")))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Calcula a mano la inversa de $\begin{bmatrix}4&3\\1&1\end{bmatrix}$ y escríbela como texto, renglón por renglón.

In [ ]:
mi_inversa = None     # escribe tu resultado como texto, por ejemplo: "1, 0; 0, 1" (se aceptan fracciones como 2/3)

if mi_inversa is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mia = matriz(str(mi_inversa), "tu inversa")
    except ValueError as err:
        print("Revisa tu matriz:", err)
    else:
        A = matriz("4, 3; 1, 1")
        ref, _ = inversa(A)
        print("La calculadora da:")
        muestra("A⁻¹", ref)
        print(compara(mia, ref, "Intercambia a y d, cambia el signo de b y c y divide entre det A = ad - bc."))
        if mia.shape == A.shape:
            print("Tu matriz por A da la identidad." if es_identidad(A * mia) else "Tu matriz por A no da la identidad.")

## 4. Interpretación geométrica del determinante

Una matriz $A$ de $2\times 2$ transforma cada punto $\mathbf x$ del plano en $A\mathbf x$. Las **columnas** de $A$ son las imágenes de los vectores de la base:

$$A=\begin{bmatrix}a_{11}&a_{12}\\a_{21}&a_{22}\end{bmatrix},\qquad A\hat{\mathbf i}=\begin{bmatrix}a_{11}\\a_{21}\end{bmatrix},\qquad A\hat{\mathbf j}=\begin{bmatrix}a_{12}\\a_{22}\end{bmatrix}.$$

- El cuadrado unitario (lados $\hat{\mathbf i}$ y $\hat{\mathbf j}$) se transforma en el paralelogramo de lados $A\hat{\mathbf i}$ y $A\hat{\mathbf j}$, de área $|\det A|$. Cualquier figura cambia su área por el mismo factor $|\det A|$.
- Si $\det A>0$, la orientación se **conserva**: lo que se recorría en sentido antihorario sigue en sentido antihorario.
- Si $\det A<0$, la orientación se **invierte**: la figura queda reflejada, como en un espejo (una letra F se ve al revés).
- Si $\det A=0$, el plano se **aplasta** sobre una recta (o sobre un punto, si $A$ es la matriz cero): el área de la imagen es 0 y la transformación no se puede deshacer, por eso $A$ no tiene inversa.

En el espacio, el volumen del paralelepípedo de aristas $\mathbf a$, $\mathbf b$, $\mathbf c$ es el valor absoluto del determinante que las tiene como columnas, que es el triple producto escalar:

$$V=\left|\det\,[\,\mathbf a\ \ \mathbf b\ \ \mathbf c\,]\right|=\left|\mathbf a\cdot(\mathbf b\times\mathbf c)\right|.$$

Mueve las entradas de $A$ con los controles. La figura original aparece punteada y su imagen rellena; un triángulo hueco marca el punto $(1,0)$, la punta de $\hat{\mathbf i}$, y el triángulo naranja marca su imagen, la punta de $A\hat{\mathbf i}$, para que veas si la figura dio la vuelta.

In [ ]:
_FIGURAS = {
    "cuadrado unitario": [(0, 0), (1, 0), (1, 1), (0, 1)],
    "letra F": [(0, 0), (0.2, 0), (0.2, 0.45), (0.5, 0.45), (0.5, 0.6), (0.2, 0.6), (0.2, 0.85), (0.6, 0.85), (0.6, 1), (0, 1)],
}
_AZUL, _NARANJA, _VERDE = "#2a78d6", "#eb6834", "#1baf7a"


def efecto(A):
    """Efecto de una matriz 2×2 sobre el plano: det, factor de área |det|, orientación e imagen del plano."""
    A = como_matriz(A, "A")
    if A.shape != (2, 2):
        raise ValueError(f"esta sección usa matrices 2×2 (esta es {dims(A)}); para tres vectores usa volumen(a, b, c)")
    d = A.det()
    if d > 0:
        orientacion = "conserva"
    elif d < 0:
        orientacion = "invierte"
    else:
        orientacion = "aplasta"
    imagen_plano = "plano" if d != 0 else ("recta" if A.rank() == 1 else "punto")
    return {"det": d, "area": abs(d), "orientacion": orientacion, "imagen": imagen_plano}


def volumen(a, b, c):
    """|a·(b×c)|: volumen exacto del paralelepípedo de aristas a, b, c."""
    vs = [vector(v, nombre) for v, nombre in ((a, "a"), (b, "b"), (c, "c"))]
    for v, nombre in zip(vs, "abc"):
        if v.rows != 3:
            raise ValueError(f"{nombre} debe tener 3 componentes (tiene {v.rows})")
    a, b, c = vs
    return abs(a.dot(b.cross(c)))


def imagen(A, puntos):
    """Imagen A·p de cada punto p = (x, y), como arreglo de numpy (para graficar)."""
    M = np.array(como_matriz(A, "A").tolist(), dtype=float)
    return np.asarray(puntos, dtype=float) @ M.T


def area_poligono(puntos):
    """Área con signo (fórmula del cordón de zapato): positiva si los vértices van en sentido antihorario."""
    P = np.asarray(puntos, dtype=float)
    xs, ys = P[:, 0], P[:, 1]
    return 0.5 * float(np.dot(xs, np.roll(ys, -1)) - np.dot(np.roll(xs, -1), ys))


def _corto(v, parentesis=False):
    """Número exacto con a lo más dos decimales (los controles avanzan de 0.1 en 0.1); negativos entre paréntesis si se pide."""
    texto = f"{float(v):.2f}".rstrip("0").rstrip(".") if v != 0 else "0"
    return f"({texto})" if parentesis and v < 0 else texto


def calculadora_geometria(a11, a12, a21, a22, figura):
    A = como_matriz([[round(a11, 1), round(a12, 1)], [round(a21, 1), round(a22, 1)]], "A")
    e = efecto(A)
    P = np.array(_FIGURAS[figura], dtype=float)
    Q = imagen(A, P)
    i_img, j_img = imagen(A, [(1, 0), (0, 1)])
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    cerrar = lambda R: np.vstack([R, R[:1]])
    ax.plot(*cerrar(P).T, "--", color="0.45", lw=1.5, label=f"{figura} (original)")
    if e["imagen"] == "plano":
        ax.fill(*Q.T, color=_AZUL, alpha=0.3, lw=0)
        ax.plot(*cerrar(Q).T, color=_AZUL, lw=2, label="imagen bajo A")
    elif e["imagen"] == "recta":
        ax.plot(*cerrar(Q).T, color=_AZUL, lw=3, label="imagen: un segmento")
    else:
        ax.plot(0, 0, "o", color=_AZUL, ms=9, label="imagen: un punto")
    for v, color, texto in ((i_img, _NARANJA, "Aî"), (j_img, _VERDE, "Aĵ")):
        if np.hypot(*v) > 1e-12:
            ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=color, lw=1.8))
            ax.annotate(texto, xy=v, xytext=(6, 6), textcoords="offset points", fontsize=10)
    ax.plot(1, 0, "^", ms=10, mfc="none", mec="0.45", ls="none", label="punto (1, 0) = punta de î")
    ax.plot(*i_img, "^", ms=11, color=_NARANJA, ls="none", label="imagen de (1, 0) = punta de Aî")
    lim = 1.15 * max(1.0, float(np.abs(np.vstack([P, Q, [i_img, j_img]])).max()))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
    ax.grid(alpha=0.3); ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(f"det A = {_corto(e['det'])}")
    ax.legend(loc="best", fontsize=8)
    plt.show()
    c = [[_corto(A[i, j]) for j in range(2)] for i in range(2)]
    p = [[_corto(A[i, j], parentesis=True) for j in range(2)] for i in range(2)]
    print(f"A = [{c[0][0]}, {c[0][1]}; {c[1][0]}, {c[1][1]}]   (columnas: Aî = ({c[0][0]}, {c[1][0]}), Aĵ = ({c[0][1]}, {c[1][1]}))")
    print(f"det A = {p[0][0]}·{p[1][1]} - {p[0][1]}·{p[1][0]} = {_corto(e['det'])}")
    if e["orientacion"] == "aplasta":
        if e["imagen"] == "recta":
            print("det A = 0: el plano se aplasta sobre una recta y la imagen de la figura es un segmento (área 0).")
        else:
            print("det A = 0 y A es la matriz cero: todo el plano cae en el origen; la imagen es un punto (área 0).")
        print("Como dos puntos distintos llegan al mismo lugar, A no se puede deshacer: no tiene inversa.")
        return
    print(f"Factor de área: |det A| = {_corto(e['area'])}. Área de la figura: {cifras(abs(area_poligono(P)), 4)} → "
          f"área de su imagen: {cifras(abs(area_poligono(Q)), 4)} (= {_corto(e['area'])} × {cifras(abs(area_poligono(P)), 4)}).")
    if e["orientacion"] == "conserva":
        print("Orientación: se conserva (det A > 0); el recorrido antihorario de la figura sigue siendo antihorario.")
    else:
        print("Orientación: se invierte (det A < 0); la figura queda reflejada, como vista en un espejo.")


_control = lambda valor, nombre: widgets.FloatSlider(value=valor, min=-3, max=3, step=0.1, description=nombre,
                                                     continuous_update=False, readout_format=".1f")
widgets.interact(calculadora_geometria,
    a11=_control(2, "a11"), a12=_control(1, "a12"), a21=_control(1, "a21"), a22=_control(1, "a22"),
    figura=widgets.Dropdown(options=list(_FIGURAS), value="cuadrado unitario", description="figura"));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
_G = matriz("-1, 0.5; 0.3, 2")
PRUEBAS_4 = [
    ("[[2,1],[1,1]]: det 1, conserva", lambda: (lambda e: e["det"] == 1 and e["orientacion"] == "conserva")(efecto(matriz("2, 1; 1, 1")))),
    ("[[1,2],[2,4]]: det 0, aplasta sobre una recta",
     lambda: (lambda e: e["det"] == 0 and e["orientacion"] == "aplasta" and e["imagen"] == "recta")(efecto(matriz("1, 2; 2, 4")))),
    ("[[-1,0],[0,1]]: det -1, invierte", lambda: (lambda e: e["det"] == -1 and e["orientacion"] == "invierte")(efecto(matriz("-1, 0; 0, 1")))),
    ("[[1.2,0.4],[0,0.8]]: factor de área 0.96 exacto", lambda: efecto(matriz("1.2, 0.4; 0, 0.8"))["area"] == sp.Rational(24, 25)),
    ("matriz cero: la imagen es un punto", lambda: efecto(matriz("0, 0; 0, 0"))["imagen"] == "punto"),
    ("volumen((2,0,0),(0,3,0),(0,0,4)) = 24", lambda: volumen("2, 0, 0", "0, 3, 0", "0, 0, 4") == 24),
    ("volumen((1,0,0),(1,1,0),(1,1,1)) = 1", lambda: volumen((1, 0, 0), (1, 1, 0), (1, 1, 1)) == 1),
    ("área de la F = 0.305 y área con signo de su imagen = det A × 0.305",
     lambda: cerca(area_poligono(_FIGURAS["letra F"]), 0.305)
             and cerca(area_poligono(imagen(_G, _FIGURAS["letra F"])), float(efecto(_G)["det"]) * 0.305)),
    ("efecto rechaza una 3×3; volumen rechaza vectores de 2 componentes",
     lambda: _rechaza(lambda: efecto(matriz("1, 0, 0; 0, 1, 0; 0, 0, 1"))) and _rechaza(lambda: volumen("1, 0", "0, 1", "1, 1"))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano el área de la imagen del cuadrado unitario bajo $A=\begin{bmatrix}3&1\\1&2\end{bmatrix}$ y escribe el número.

In [ ]:
mi_area = None     # escribe un número (también sirve como texto, por ejemplo "2.5")

if mi_area is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = numero(mi_area, "tu área")
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = efecto(matriz("3, 1; 1, 2"))["area"]
        print("La calculadora da: área =", ref)
        print("coinciden" if mio == ref else
              "NO coinciden: el cuadrado unitario tiene área 1, así que su imagen tiene área |det A| = |a₁₁a₂₂ - a₁₂a₂₁|")

## 5. Factorización LU

En el libro esta sección es una ampliación opcional. La eliminación gaussiana, sin intercambiar renglones, escribe una matriz cuadrada como $A=LU$ (factorización de Doolittle):

- $U$ es **triangular superior**: es la matriz que queda al final de la eliminación.
- $L$ es **triangular inferior con unos en la diagonal** y guarda los multiplicadores: en el paso $k$, al renglón $i$ le restas $\ell_{ik}$ veces el renglón del pivote, con $\ell_{ik}=\dfrac{\text{entrada }(i,k)\text{ en ese paso}}{\text{pivote }u_{kk}}$, y ese $\ell_{ik}$ queda en la posición $(i,k)$ de $L$.
- Como $\det L=1$, el determinante sale gratis: $\det A=\det U=u_{11}u_{22}\cdots u_{nn}$, con unas $n^3/3$ multiplicaciones en lugar de las $\approx 1.7\,n!$ de los cofactores.

Para resolver $A\mathbf x=\mathbf b$ se hacen dos sustituciones:

$$L\mathbf y=\mathbf b\ \ (\text{hacia adelante: } y_1, y_2, \dots),\qquad U\mathbf x=\mathbf y\ \ (\text{hacia atrás: } x_n, x_{n-1}, \dots).$$

La ventaja aparece cuando la misma $A$ se usa con muchos $\mathbf b$: se factoriza una sola vez y cada $\mathbf b$ nuevo cuesta solo las dos sustituciones.

Si en algún paso el pivote $u_{kk}$ vale 0 y hay entradas distintas de cero debajo de él, hay que **intercambiar renglones**; con la matriz de permutación $P$ que registra los intercambios queda $PA=LU$. Eso es lo que hacen `numpy` y `scipy` (que además intercambian renglones para usar el pivote más grande y reducir el error de redondeo). Esta calculadora no intercambia renglones: te avisa en qué paso haría falta.

En la caja de $\mathbf b$ puedes escribir uno o varios lados derechos separados con `;`, por ejemplo `10, 20, 30; 0, 0, 10`.

In [ ]:
def lu_sin_pivoteo(A):
    """(L, U) exactas con A = LU (Doolittle: L con unos en la diagonal, sin intercambiar renglones)."""
    A = como_matriz(A, "A")
    _cuadrada(A, "aquí la factorización LU es solo para matrices cuadradas")
    n = A.rows
    L, U = sp.eye(n), A.copy()
    for k in range(n - 1):
        if U[k, k] == 0:
            if any(U[i, k] != 0 for i in range(k + 1, n)):
                raise ValueError(f"pivote cero en el paso {k + 1}: hace falta intercambiar renglones (PA = LU)")
            continue                                   # debajo del pivote ya hay ceros: no hay nada que eliminar
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]
            L[i, k] = m
            U[i, :] = U[i, :] - m * U[k, :]
    return L, U


def det_triangular(T):
    """Determinante de una matriz triangular: el producto de su diagonal."""
    T = como_matriz(T, "U")
    _cuadrada(T)
    if not (T.is_upper or T.is_lower):
        raise ValueError("det_triangular necesita una matriz triangular")
    return sp.prod([T[k, k] for k in range(T.rows)])


def _revisa_triangular(T, b, letra, inferior):
    T, b = como_matriz(T, letra), vector(b, "b" if letra == "L" else "y")
    _cuadrada(T, f"{letra} debe ser cuadrada")
    if b.rows != T.rows:
        raise ValueError(f"el vector tiene {_entradas(b.rows)} y {letra} tiene {T.rows} renglones: deben coincidir")
    if inferior and not T.is_lower:
        raise ValueError("L debe ser triangular inferior (ceros arriba de la diagonal)")
    if not inferior and not T.is_upper:
        raise ValueError("U debe ser triangular superior (ceros debajo de la diagonal)")
    return T, b


def adelante(L, b):
    """Sustitución hacia adelante: resuelve L y = b (L triangular inferior)."""
    L, b = _revisa_triangular(L, b, "L", inferior=True)
    n = L.rows
    y = sp.zeros(n, 1)
    for i in range(n):
        if L[i, i] == 0:
            raise ValueError(f"l{_sub(i + 1, i + 1)} = 0: no se puede despejar y{_sub(i + 1)}")
        y[i] = (b[i] - sum((L[i, j] * y[j] for j in range(i)), sp.Integer(0))) / L[i, i]
    return y


def atras(U, y):
    """Sustitución hacia atrás: resuelve U x = y (U triangular superior)."""
    U, y = _revisa_triangular(U, y, "U", inferior=False)
    n = U.rows
    x = sp.zeros(n, 1)
    for i in reversed(range(n)):
        if U[i, i] == 0:
            raise ValueError(f"u{_sub(i + 1, i + 1)} = 0 en la diagonal de U: det A = 0 y el sistema no tiene solución única")
        x[i] = (y[i] - sum((U[i, j] * x[j] for j in range(i + 1, n)), sp.Integer(0))) / U[i, i]
    return x


def resuelve_lu(L, U, b):
    """(y, x): primero L y = b hacia adelante, luego U x = y hacia atrás."""
    y = adelante(L, b)
    return y, atras(U, y)


def lados_derechos(texto, n):
    """Lista de vectores b de n entradas: uno por renglón ('10, 20, 30; 0, 0, 10') o uno solo escrito en columna."""
    try:
        B = matriz(texto, "b")
    except ValueError as err:
        if str(err).startswith("escribe "):
            raise ValueError("escribe b como lista de números, por ejemplo: 1, 2, 3 (varios b se separan con ;)") from None
        raise
    if B.cols == 1 and B.rows == n and n > 1:
        return [B]                                     # un solo b escrito en columna: 10; 20; 30
    if B.cols != n:
        raise ValueError(f"cada b debe tener {_entradas(n)}, tantas como renglones de A (escribiste {_entradas(B.cols)})")
    return [B.row(k).T for k in range(B.rows)]


def _numpy_resuelve(A, b, n_cifras):
    if A.det() == 0:          # en punto flotante, solve puede "resolver" una matriz singular y dar números sin sentido
        return "numpy.linalg.solve no aplica: det A = 0 (en punto flotante podría devolver números sin sentido sin avisar)"
    try:
        xs = np.linalg.solve(np.array(A.tolist(), dtype=float), np.array(b.tolist(), dtype=float).ravel())
    except np.linalg.LinAlgError:
        return "numpy.linalg.solve: la matriz es singular (det A = 0)"
    xs = np.where(np.abs(xs) < 1e-12 * max(1.0, float(np.abs(xs).max())), 0.0, xs)   # residuos de redondeo como 1e-16 -> 0
    return "numpy.linalg.solve (con intercambio de renglones, en punto flotante): x ≈ (" + ", ".join(cifras(v, n_cifras) for v in xs) + ")"


def calculadora_lu(A_txt, b_txt, n_cifras):
    try:
        A = matriz(A_txt, "A")
        _cuadrada(A, "aquí la factorización LU es solo para matrices cuadradas")
        bs = lados_derechos(b_txt, A.rows)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    muestra("A", A, n_cifras)
    try:
        L, U = lu_sin_pivoteo(A)
    except ValueError as err:
        print("Sin intercambiar renglones no se puede:", err)
        if A.det() == 0:
            print("Además det A = 0: ni intercambiando renglones hay solución única.")
            return
        print("numpy sí resuelve el sistema porque intercambia renglones (PA = LU):")
        for k, b in enumerate(bs, 1):
            print(f"  b{_sub(k)} = {_tupla(b)}:", _numpy_resuelve(A, b, n_cifras))
        return
    muestra("L", L, n_cifras)
    muestra("U", U, n_cifras)
    print("Verificación: LU = A" if L * U == A else "Verificación FALLÓ: LU ≠ A")
    d, n = det_triangular(U), U.rows
    nombres = "·".join(f"u{_sub(k, k)}" for k in range(1, n + 1)) if n <= 3 else f"u₁₁·…·u{_sub(n, n)}"
    cuenta = f" = {'·'.join(_paren(U[k, k]) for k in range(n))}" if n > 1 else ""
    print(f"det A = {nombres}{cuenta} = {d}" + ("" if d.is_integer else f" ≈ {cifras(d, n_cifras)}"))
    for k, b in enumerate(bs, 1):
        print(f"\n— Lado derecho b{_sub(k)} = {_tupla(b)} —")
        try:
            y, x = resuelve_lu(L, U, b)
        except ValueError as err:
            print("No se puede despejar:", err)
            print(_numpy_resuelve(A, b, n_cifras))
            continue
        print("L y = b (hacia adelante):  y =", _tupla(y, n_cifras))
        print("U x = y (hacia atrás):     x =", _tupla(x, n_cifras))
        print("Verificación: A x = b" if A * x == b else "Verificación FALLÓ: A x ≠ b")
        print(_numpy_resuelve(A, b, n_cifras))


widgets.interact(calculadora_lu,
    A_txt=widgets.Textarea(value="4, -2, 0\n-2, 4, -2\n0, -2, 2", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="320px", height="110px")),
    b_txt=widgets.Textarea(value="10, 20, 30; 0, 0, 10; 5, 5, 5", description="b =", continuous_update=False,
                           layout=widgets.Layout(width="320px", height="60px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
_K = matriz("4, -2, 0; -2, 4, -2; 0, -2, 2")
_LK, _UK = lu_sin_pivoteo(_K)
_A4 = matriz("1, 2, 1, 0; 2, 5, 3, 1; 1, 3, 4, 2; 0, 1, 2, 5")
PRUEBAS_5 = [
    ("L = [[1,0,0],[-1/2,1,0],[0,-2/3,1]] y U = [[4,-2,0],[0,3,-2],[0,0,2/3]]",
     lambda: _LK == matriz("1, 0, 0; -1/2, 1, 0; 0, -2/3, 1") and _UK == matriz("4, -2, 0; 0, 3, -2; 0, 0, 2/3")),
    ("det A desde U: 4·3·(2/3) = 8", lambda: det_triangular(_UK) == 8 == _K.det()),
    ("b = (10, 20, 30) -> x = (30, 55, 70)", lambda: resuelve_lu(_LK, _UK, "10, 20, 30")[1] == vector("30, 55, 70")),
    ("b = (0, 0, 10) -> x = (5, 10, 15)", lambda: resuelve_lu(_LK, _UK, "0, 0, 10")[1] == vector("5, 10, 15")),
    ("b = (5, 5, 5) -> x = (15/2, 25/2, 15)", lambda: resuelve_lu(_LK, _UK, "5, 5, 5")[1] == vector("15/2, 25/2, 15")),
    ("[[0,1],[1,0]] se rechaza: pivote cero en el paso 1",
     lambda: _mensaje(lambda: lu_sin_pivoteo(matriz("0, 1; 1, 0"))).startswith("pivote cero en el paso 1")),
    ("pivote cero a la mitad (paso 2 de una 3×3) también se rechaza",
     lambda: "paso 2" in _mensaje(lambda: lu_sin_pivoteo(matriz("1, 2, 3; 2, 4, 5; 1, 1, 1")))),
    ("LU = A para una 4×4 (L con unos en la diagonal, U triangular superior)",
     lambda: (lambda L, U: L * U == _A4 and L.is_lower and U.is_upper and all(L[k, k] == 1 for k in range(4)))(*lu_sin_pivoteo(_A4))),
    ("varios b separados con ';' y un b escrito en columna",
     lambda: len(lados_derechos("10, 20, 30; 0, 0, 10", 3)) == 2 and lados_derechos("10; 20; 30", 3)[0] == vector("10, 20, 30")),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Factoriza a mano $A=\begin{bmatrix}3&1\\6&5\end{bmatrix}=LU$ (sin intercambiar renglones) y escribe $L$ y $U$ como texto.

In [ ]:
mi_L = None     # escribe L como texto, por ejemplo: "1, 0; 5, 1"
mi_U = None     # escribe U como texto, por ejemplo: "3, 2; 0, 7"

if mi_L is None or mi_U is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mia_L, mia_U = matriz(str(mi_L), "tu L"), matriz(str(mi_U), "tu U")
    except ValueError as err:
        print("Revisa tu matriz:", err)
    else:
        ref_L, ref_U = lu_sin_pivoteo(matriz("3, 1; 6, 5"))
        print("La calculadora da:")
        muestra("L", ref_L)
        muestra("U", ref_U)
        print("L:", compara(mia_L, ref_L, "El multiplicador es ℓ₂₁ = a₂₁/a₁₁ y va debajo del 1 de la diagonal."))
        print("U:", compara(mia_U, ref_U, "U es lo que queda al restar ℓ₂₁ veces el renglón 1 al renglón 2."))

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")